# Embedding models

In [ ]:
import pandas as pd
comment_df = pd.read_csv(r'\data\non_freebie_comments.csv')

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

# ---------------------------------------
# 1. Load model
# ---------------------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
chris_comments = comment_df[comment_df.account_handle=='chrislovesjulia']
# ---------------------------------------
# 2. Prepare comments
# ---------------------------------------
comments = chris_comments["comment_clean"].fillna("").astype(str).tolist()

# ---------------------------------------
# 3. Compute embeddings
# ---------------------------------------
embeddings = model.encode(
    comments,
    batch_size=64,
    show_progress_bar=True
)

# ---------------------------------------
# 4. Cluster into 15 groups
# ---------------------------------------
num_clusters = 15
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

# ---------------------------------------
# 5. Attach cluster labels back to dataframe
# ---------------------------------------
chris_comments["cluster"] = labels

# Optional: inspect cluster examples
for cluster_id in range(num_clusters):
    print(f"\nCluster {cluster_id}")
    sample = comment_df[comment_df["cluster"] == cluster_id]["comment_clean"].head(5)
    print(sample.to_list())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]


Cluster 0
['can you share', 'yessss', 'good', 'no maam', 'jajaja ??']

Cluster 1
['so great!! ??', '??????brilliant', 'love it ??', 'i love it! ??', 'seriously so good!! ??']

Cluster 2
['totally or match the house vibe', 'i agree!!!', 'agreed!!', 'team black pool here too !!', 'same girl same!!!']

Cluster 3
['i agree', 'people in dc should take some notes', 'i agree', '100 agree', 'and this also applies to national monuments ??']

Cluster 4
['i would like to add it takes two and this list is perfect????', '5', '5!!', '6! we loved building those for you and your team', '7']

Cluster 5
['your pool is gorgeous period', 'gorgeous space ??', 'this looks so much better anyway', 'these are pretty neat ??', 'still photos of before and after please']

Cluster 6
['actually do whatever you want and enjoy yourself ??', 'nope i like to see what lurks below ??', 'i love obsessing over these kinds of details! ??', 'i love seeing people have awesome stuff and i remain poor with my toast and butter'

/tmp/ipykernel_1689/3739035334.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chris_comments["cluster"] = labels


In [ ]:
chris_comments.cluster.value_counts()

,count
cluster,
1,289
9,203
5,201
0,183
14,151
3,149
13,149
6,145
12,142


In [ ]:
comment_df['comment_demands'].isna().sum() - 20615

np.int64(-3552)

In [ ]:
chris_comments['comment_category'].value_counts()

,count
comment_category,
Other,1408
Request,658
Pain Point,66
Purchase Intent,17
Confusion,9


In [ ]:
# classification
requests = comment_df[(comment_df['comment_category']=='Request')&(comment_df.has_request==True)]
pain_points = comment_df[(comment_df['comment_category']=='Pain Point')&(comment_df.has_pain_point==True)]
confusion = comment_df[(comment_df['comment_category']=='Confusion')&(comment_df.has_confusion==True)]
purchase = comment_df[(comment_df['comment_category']=='Purchase Intent')&(comment_df.has_buy_intent==True)]
other = comment_df[(comment_df['comment_category']!='Request')
                    &(comment_df['comment_category']!='Pain Point')
                    &(comment_df['comment_category']!='Confusion')
                    &(comment_df['comment_category']!='Purchase Intent')
                    &(comment_df.has_request!=True)
                    &(comment_df.has_pain_point!=True)
                    &(comment_df.has_confusion!=True)
                    &(comment_df.has_buy_intent!=True)].head(100)
training_df = pd.concat([requests, pain_points, confusion, purchase,other],axis=0).reset_index().drop('index',axis=1)


In [ ]:
comment_df.columns

Index(['media_id', 'assigned_niche', 'account_handle', 'timestamp', 'caption',
       'comment_user', 'comment_text', 'comment_clean', 'positive_jerry',
       'positive_nltk', 'comment_for_freebie', 'comment_link',
       'is_question_marks', 'comment_topics', 'comment_demands',
       'comment_category', 'comment_emotions', 'caption_keywords',
       'comment_keywords', 'comment_length', 'word_count', 'is_question',
       'has_request', 'has_buy_intent', 'has_confusion', 'has_pain_point',
       'has_positive_reaction', 'has_negative_reaction', 'sentiment_score',
       'sentiment_bucket', 'cluster'],
      dtype='object')

In [ ]:
training_df = training_df[['comment_clean','comment_category']]
training_df.head()

,comment_clean,comment_category
0,can you share,Request
1,can you share this with the white house? ??,Request
2,love the set up! can you send me more info on ...,Request
3,can you share the link thk u,Request
4,finally a person that knows how to not over do...,Request


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Load embedding model
# ---------------------------------------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# ---------------------------------------------------------
# 2. Prepare training data
# ---------------------------------------------------------
train_texts = training_df["comment_clean"].fillna("").astype(str).tolist()
train_labels = training_df["comment_category"].astype(str).tolist()

# ---------------------------------------------------------
# 3. Embed training comments
# ---------------------------------------------------------
train_embeddings = model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True
)

# ---------------------------------------------------------
# 4. Train classifier on embeddings
# ---------------------------------------------------------
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    multi_class="auto"
)
clf.fit(train_embeddings, train_labels)

# ---------------------------------------------------------
# 5. Prepare full comment set to classify
# ---------------------------------------------------------
all_comments = comment_df["comment_clean"].fillna("").astype(str).tolist()

# ---------------------------------------------------------
# 6. Embed all comments
# ---------------------------------------------------------
all_embeddings = model.encode(
    all_comments,
    batch_size=64,
    show_progress_bar=True
)

# ---------------------------------------------------------
# 7. Predict categories
# ---------------------------------------------------------
predicted_labels = clf.predict(all_embeddings)

# ---------------------------------------------------------
# 8. Attach predictions to dataframe
# ---------------------------------------------------------
comment_df["predicted_category"] = predicted_labels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Batches:   0%|          | 0/323 [00:00<?, ?it/s]

In [ ]:
comment_df['predicted_category'].value_counts()

,count
predicted_category,
Request,7127
Other,6401
Pain Point,3467
Confusion,2711
Purchase Intent,909


In [ ]:
comment_df.to_csv(r'\data\comparison.csv',index=False)